# Bookeau · Modelo dimensional y carga Gold

Este cuaderno consulta `data/oro/bookeau.sqlite3`, construido desde Silver. R01 conserva estados originales y agrupa atención. R02 deja fuera de Gold las encuestas excluidas.

Resultados de la carga: 86.562 reservas, 60.224 encuestas incluidas y 384.792 respuestas no vacías. Se conciliaron granos y claves durante la carga. Las celdas del cuaderno todavía no se han ejecutado en Jupyter.

Modelo y diagrama: `docs/modelo/modelo_dimensional.md`. Arquitectura: `docs/arquitectura/ecosistema_analitica.md`.


In [1]:
from pathlib import Path
import sqlite3
import json
from IPython.display import display, Markdown

PROYECTO = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "docs" / "plan_entrega_1.md").is_file())
db = sqlite3.connect(f"file:{PROYECTO / 'data/oro/bookeau.sqlite3'}?mode=ro", uri=True)
db.row_factory = sqlite3.Row

def consulta(sql):
    filas = [dict(r) for r in db.execute(sql)]
    if not filas:
        print("Sin filas")
        return
    columnas = list(filas[0])
    esc = lambda x: str(x).replace("|", "\|").replace("\n", " ")
    tabla = ["| " + " | ".join(columnas) + " |", "| " + " | ".join(["---"]*len(columnas)) + " |"]
    tabla.extend("| " + " | ".join(esc(fila[c]) for c in columnas) + " |" for fila in filas)
    display(Markdown("\n".join(tabla)))


<>:17: SyntaxWarning: invalid escape sequence '\|'
<>:17: SyntaxWarning: invalid escape sequence '\|'
/var/folders/jq/9m84jg7x2hz0l32cqnwklfbm0000gn/T/ipykernel_46515/3704755985.py:17: SyntaxWarning: invalid escape sequence '\|'
  esc = lambda x: str(x).replace("|", "\|").replace("\n", " ")


## 1. Controles de la carga


In [2]:
controles = json.loads((PROYECTO / "docs" / "transformacion" / "controles_gold.json").read_text())
print(json.dumps(controles, ensure_ascii=False, indent=2))
consulta("SELECT estado_original, estado_analitico FROM dim_estado")


{
  "conteos": {
    "dim_fecha": 3565,
    "dim_hora": 1440,
    "dim_periodo": 29,
    "dim_servicio": 11,
    "dim_modalidad": 24,
    "dim_estado": 12,
    "dim_programa": 175,
    "dim_tipo_encuesta": 4,
    "dim_pregunta": 22,
    "hecho_reserva": 86562,
    "hecho_encuesta": 60224,
    "hecho_respuesta": 384792
  },
  "reservas_silver": 86562,
  "reservas_gold": 86562,
  "atendidas_R01": 42646,
  "encuestas_excluidas_R02": 166,
  "claves_foraneas": "OK",
  "unicidad_granos": "OK",
  "conciliacion": "OK"
}


| estado_original | estado_analitico |
| --- | --- |
| Cancelada | Cancelada |
| Cancelada a tiempo | Cancelada a tiempo |
| Cancelada con excusa | Cancelada con excusa |
| Cancelada por sanción | Cancelada por sanción |
| Cancelada y sancionada | Cancelada y sancionada |
| Cola cancelada por reservación | Cola cancelada por reservación |
| En cola | En cola |
| En ejecución | En ejecución |
| Finalizada | Atendida |
| No asistió | No asistió |
| Realizada | Atendida |
| Reservada | Reservada |

## 2. Reserva y atención registrada

No hay oferta general disponible; los eventos no miden ocupación. Las medidas se agregan directamente desde hecho_reserva.


In [3]:
consulta("""
SELECT p.periodo_original, COUNT(*) eventos,
       SUM(CASE WHEN e.estado_analitico='Atendida' THEN 1 ELSE 0 END) atenciones
FROM hecho_reserva h JOIN dim_periodo p USING(sk_periodo)
JOIN dim_estado e USING(sk_estado)
GROUP BY p.periodo_original ORDER BY p.periodo_original
""")


| periodo_original | eventos | atenciones |
| --- | --- | --- |
| 201620 | 2846 | 1308 |
| 201710 | 3930 | 2264 |
| 201719 | 80 | 48 |
| 201720 | 3372 | 1691 |
| 201810 | 3821 | 1910 |
| 201819 | 132 | 61 |
| 201820 | 4002 | 1944 |
| 201910 | 4495 | 1904 |
| 201919 | 106 | 61 |
| 201920 | 3471 | 1511 |
| 202010 | 4287 | 1876 |
| 202019 | 240 | 128 |
| 202020 | 5248 | 3023 |
| 202110 | 6077 | 3304 |
| 202119 | 110 | 52 |
| 202120 | 4596 | 2596 |
| 202210 | 5163 | 2371 |
| 202219 | 102 | 55 |
| 202220 | 5563 | 2414 |
| 202310 | 5487 | 2660 |
| 202319 | 219 | 93 |
| 202320 | 5430 | 2522 |
| 202410 | 4990 | 2334 |
| 202419 | 147 | 76 |
| 202420 | 4870 | 2294 |
| 202510 | 3452 | 1803 |
| 202519 | 38 | 22 |
| 202520 | 2318 | 1190 |
| 202610 | 1970 | 1131 |

## 3. Satisfacción

Solo encuestas posteriores incluidas; el denominador de completitud es encuesta incluida, no población invitada. La media es descriptiva y debe acompañarse de n y distribución.


In [4]:
consulta("""
SELECT m.tipo_horario_original, COUNT(*) encuestas,
       COUNT(h.calificacion_ayuda_tutor) n_calificaciones,
       ROUND(AVG(h.calificacion_ayuda_tutor),4) media_descriptiva
FROM hecho_encuesta h JOIN dim_tipo_encuesta t USING(sk_tipo_encuesta)
JOIN dim_modalidad m USING(sk_modalidad)
WHERE t.etapa='posterior' GROUP BY m.tipo_horario_original
""")


| tipo_horario_original | encuestas | n_calificaciones | media_descriptiva |
| --- | --- | --- | --- |
| Deprecated, Grupal | 60 | 60 | 4.6 |
| Deprecated, Torre Séneca | 87 | 61 | 4.7049 |
| Express | 6646 | 4878 | 4.6982 |
| Normal | 14731 | 9722 | 4.7725 |
| Normal pico | 1450 | 1450 | 4.6241 |

## 4. Reconstruir Gold y tablero

La reconstrucción usa Silver. Cerrar la conexión antes de reemplazar la base. El tablero local se genera sin publicar nada fuera del equipo.


In [5]:
# db.close()
# import runpy, sys
# sys.path.insert(0, str(PROYECTO / "src"))
# runpy.run_path(str(PROYECTO / "src" / "construir_gold_bookeau.py"), run_name="__main__")
# runpy.run_path(str(PROYECTO / "src" / "generar_tablero_bookeau.py"), run_name="__main__")
